In [47]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def preprocess(train_file, test_file):
    train = np.load(train_file)
    test = np.load(test_file)

    X_train, y_train = train["X"], train["y"]
    print(X_train.shape, y_train.shape)
    X_test, y_test = test["X"], test["y"]
    print(X_test.shape, y_test.shape)

    print(len(np.unique(y_train)))

    X_train = torch.as_tensor(data=X_train, dtype=torch.float32)
    X_test  = torch.as_tensor(data=X_test,  dtype=torch.float32) 

    mean = torch.mean(input=X_train, dim=0)
    std  = torch.std(input=X_train, dim=0)

    X_train = (X_train - mean) / std
    X_test  = (X_test  - mean) / std

    feats = 32
    n = X_train.shape[0]
    K = (X_train.T @ X_train) / (n - 1)
    print(K)

    D, V = torch.linalg.eigh(K)
    idx  = torch.argsort(input=D, descending=True)

    D = D[idx]
    V = V[:, idx]

    train_PCA = X_train @ V[:, :feats]
    test_PCA  = X_test @ V[:, :feats]

    X_train = train_PCA / torch.sqrt(input=D[:feats])
    X_test = test_PCA / torch.sqrt(input=D[:feats])

    return (X_train, y_train), (X_test, y_test)

(X_train, y_train), (X_test, y_test) = preprocess("train1.npz", "test1.npz")

(1600, 216) (1600,)
(400, 216) (400,)
10
tensor([[ 1.0000,  0.4949,  0.5580,  ..., -0.0803, -0.2078,  0.1061],
        [ 0.4949,  1.0000,  0.6267,  ..., -0.3758, -0.3044,  0.0700],
        [ 0.5580,  0.6267,  1.0000,  ..., -0.2510, -0.2791,  0.2092],
        ...,
        [-0.0803, -0.3758, -0.2510,  ...,  1.0000,  0.1222,  0.0294],
        [-0.2078, -0.3044, -0.2791,  ...,  0.1222,  1.0000, -0.0904],
        [ 0.1061,  0.0700,  0.2092,  ...,  0.0294, -0.0904,  1.0000]])


In [48]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.act1 = nn.ReLU()
        self.act2 = nn.LeakyReLU()

        self.fc1 = nn.Linear(in_features=32, out_features=16)
        self.drop1 = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(in_features=16, out_features=32)
        self.drop2 = nn.Dropout(p=0.2)
        self.fc3 = nn.Linear(in_features=32, out_features=16)
        self.drop3 = nn.Dropout(p=0.2)
        self.fc4 = nn.Linear(in_features=16, out_features=10)

    def forward(self, x):
        x = self.drop1(self.act1(self.fc1(x)))
        x = self.drop2(self.act2(self.fc2(x)))
        x = self.drop3(self.act1(self.fc3(x)))
        x = self.fc4(x)

        return x

model = Model()

In [49]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def test(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * len(y)
            correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100

best = 0
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}"
    )
    if test_acc > best:
        best = test_acc
        torch.save(model.state_dict(), "model1.pth")

Epoch 1/100 | Train Loss: 2.2987 | Train Acc: 11.81% | Test Loss: 2.2772 | Test Acc: 13.50%


/tmp/ipykernel_39163/780038797.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_train, dtype=torch.float32),
/tmp/ipykernel_39163/780038797.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_test, dtype=torch.float32),


Epoch 2/100 | Train Loss: 2.2634 | Train Acc: 16.56% | Test Loss: 2.2006 | Test Acc: 29.00%
Epoch 3/100 | Train Loss: 2.1253 | Train Acc: 30.00% | Test Loss: 1.9262 | Test Acc: 48.00%
Epoch 4/100 | Train Loss: 1.8084 | Train Acc: 40.44% | Test Loss: 1.4827 | Test Acc: 59.00%
Epoch 5/100 | Train Loss: 1.5143 | Train Acc: 46.75% | Test Loss: 1.1758 | Test Acc: 65.75%
Epoch 6/100 | Train Loss: 1.3142 | Train Acc: 54.31% | Test Loss: 0.9606 | Test Acc: 75.75%
Epoch 7/100 | Train Loss: 1.1504 | Train Acc: 61.12% | Test Loss: 0.7828 | Test Acc: 81.25%
Epoch 8/100 | Train Loss: 1.0269 | Train Acc: 65.94% | Test Loss: 0.6489 | Test Acc: 85.25%
Epoch 9/100 | Train Loss: 0.9185 | Train Acc: 68.94% | Test Loss: 0.5466 | Test Acc: 89.50%
Epoch 10/100 | Train Loss: 0.8722 | Train Acc: 71.94% | Test Loss: 0.4674 | Test Acc: 89.75%
Epoch 11/100 | Train Loss: 0.7971 | Train Acc: 73.56% | Test Loss: 0.4101 | Test Acc: 91.50%
Epoch 12/100 | Train Loss: 0.7471 | Train Acc: 75.56% | Test Loss: 0.3524 | Te